## 1. Sales performance

### Which product categories generate the highest revenue?

In [0]:
%sql
WITH category_sales AS (
    SELECT
        p.product_category_name AS category,
        SUM(ot.price) AS total_sales
    FROM workspace.ecommerce.orders o
    LEFT JOIN workspace.ecommerce.order_items ot
        ON o.order_id = ot.order_id
    LEFT JOIN workspace.ecommerce.products p
        ON p.product_id = ot.product_id
    WHERE o.order_status = 'delivered'
    GROUP BY p.product_category_name
)

SELECT
    category,
    ROUND(total_sales, 2) AS total_sales,
    ROUND(
        100.0 * total_sales / SUM(total_sales) OVER (),
        2
    ) AS sales_percentage
FROM category_sales
ORDER BY sales_percentage DESC
LIMIT 10;

category,total_sales,sales_percentage
beleza_saude,1233131.72,9.33
relogios_presentes,1166176.98,8.82
cama_mesa_banho,1023434.76,7.74
esporte_lazer,954852.55,7.22
informatica_acessorios,888724.61,6.72
moveis_decoracao,711927.69,5.38
utilidades_domesticas,615628.69,4.66
cool_stuff,610204.1,4.62
automotivo,578966.65,4.38
ferramentas_jardim,470495.28,3.56


### Which product categories have the highest demand?

In [0]:
%sql
WITH category_orders AS (
    SELECT
        p.product_category_name AS category,
        COUNT(DISTINCT o.order_id) AS total_orders
    FROM workspace.ecommerce.orders o
    LEFT JOIN workspace.ecommerce.order_items ot
        ON o.order_id = ot.order_id
    LEFT JOIN workspace.ecommerce.products p
        ON ot.product_id = p.product_id
    WHERE o.order_status = 'delivered'
    GROUP BY p.product_category_name
)

SELECT
    category,
    total_orders,
    ROUND(
        100.0 * total_orders / SUM(total_orders) OVER (),
        2
    ) AS order_percentage
FROM category_orders
ORDER BY order_percentage DESC
LIMIT 10;

category,total_orders,order_percentage
cama_mesa_banho,9272,9.53
beleza_saude,8647,8.89
esporte_lazer,7530,7.74
informatica_acessorios,6530,6.71
moveis_decoracao,6307,6.48
utilidades_domesticas,5743,5.90
relogios_presentes,5495,5.65
telefonia,4093,4.21
automotivo,3810,3.92
brinquedos,3804,3.91


## 2. Delivery Performance

### What percentage of orders are delivered on time versus late?

In [0]:
%sql

SELECT
    CASE
        WHEN order_delivered_customer_date <= order_estimated_delivery_date
        THEN 'On Time'
        ELSE 'Late'
    END AS delivery_status,
    COUNT(*) AS total_orders,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS delivery_rate
FROM orders
WHERE order_status = 'delivered'
GROUP BY 1;



delivery_status,total_orders,delivery_rate
On Time,88644,91.88
Late,7834,8.12


### Which Brazilian states have the worst delivery performance?

In [0]:
%sql
SELECT customer_state, COUNT(*) AS total_orders, 
ROUND(AVG(delivery_days),2) AS avg_delivery_days,
ROUND(100*(SUM(is_late)/ count(*)),2 ) AS late_delivery_rate
FROM workspace.ecommerce.delivery_performance 
WHERE order_status = 'delivered'
GROUP BY customer_state
ORDER BY late_delivery_rate DESC
LIMIT 10;


customer_state,total_orders,avg_delivery_days,late_delivery_rate
AL,397,24.5,23.93
MA,717,21.51,19.67
PI,476,19.39,15.97
CE,1279,21.2,15.32
SE,335,21.46,15.22
BA,3256,19.28,14.04
RJ,12350,15.24,13.47
TO,274,17.6,12.77
PA,946,23.73,12.37
ES,1995,15.72,12.23


## 3.Customer retention

### Customer retention overtime

In [0]:
%sql

SELECT ROUND(
    100 * ( COUNT(DISTINCT CASE
        WHEN  is_repeat_customer = 1
        THEN customer_unique_id
    END)/ COUNT(*)), 4
) as Overall_return_rate
FROM ecommerce.customer_retention;




Overall_return_rate
3.0003


### Customer retention by year and month

In [0]:
%sql
SELECT
    DATE_FORMAT(o.order_purchase_timestamp, 'yyyy-MM') AS purchase_date,
    COUNT(DISTINCT c.customer_unique_id) AS total_customers,
    COUNT(DISTINCT CASE
        WHEN ct.is_repeat_customer = 1
        THEN c.customer_unique_id
    END) AS returning_customers,

    ROUND(
        100.0 * (returning_customers/total_customers),2
    ) AS repeat_customer_rate

FROM workspace.ecommerce.orders o

JOIN workspace.ecommerce.customers c
    ON c.customer_id = o.customer_id

LEFT JOIN workspace.ecommerce.customer_retention ct
    ON c.customer_unique_id = ct.customer_unique_id

GROUP BY 1
ORDER BY 3 DESC, 4 DESC;


purchase_date,total_customers,returning_customers,repeat_customer_rate
2018-02,6569,348,5.3
2017-11,7430,346,4.66
2018-01,7166,340,4.74
2018-03,7115,303,4.26
2018-05,6814,297,4.36
2018-04,6882,291,4.23
2017-09,4212,259,6.15
2017-12,5603,253,4.52
2018-06,6128,253,4.13
2017-10,4561,249,5.46
